# Ordered Logistic Regression Results for Adoption Predictors of Indigenous and Modern Knowledge in Rangeland Management Practices, Northern Kenya Exploration with `mlcroissant`
This notebook provides a step-by-step guide for loading and exploring the FAIR^2 dataset using the `mlcroissant` library.

### Dataset Source
The dataset source is provided via a Croissant schema URL.

In [ ]:
# Ensure mlcroissant library is installed
!pip install mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd
import matplotlib.pyplot as plt

# Define the Croissant schema URL
url = 'https://sen.science/doi/10.71728/senscience.y7m0-f273/fair2.json'

# Load the dataset metadata
dataset = mlc.Dataset(url)
metadata = dataset.metadata
print("\nDataset Name:")
print(metadata.name)
print("\nDescription:")
print(metadata.description)
print("\nIdentifier:")
print(metadata.identifier)
print("\nKeywords:")
print(metadata.keywords)
print("\nSpatial Coverage:")
print(metadata.spatialCoverage)
print("\nTemporal Coverage:")
print(metadata.temporalCoverage)


## 2. Data Overview
Review available record sets, fields, and their IDs. Each entity is referenced by its `@id`.

In [ ]:
# Access record sets by @id
record_sets = []

# Extract all record set @ids from metadata
if hasattr(metadata, 'recordSet') and metadata.recordSet:
    for rs in metadata.recordSet:
        record_sets.append(rs['@id'])
else:
    print("No record sets found in metadata.")

# Print comprehensive overview for each record set
for rs_id in record_sets:
    print(f"\nRecord Set @id: {rs_id}")
    # Print available fields within the record set
    fields = []
    rs_obj = None
    for rs in metadata.recordSet:
        if rs['@id'] == rs_id:
            rs_obj = rs
            break
    if rs_obj:
        if 'field' in rs_obj:
            for fld in rs_obj['field']:
                fld_id = fld['@id'] if isinstance(fld, dict) and '@id' in fld else fld
                fields.append(fld_id)
        else:
            print("No fields found in this record set.")
        print("Fields @ids:", fields)
    else:
        print("Record set metadata not found.")

# Example: Show a preview of the first few records from the first available record set
if record_sets:
    for x in dataset.records(record_set=record_sets[0]):
        print(x)
        break  # Show only the first record as preview

## 3. Data Extraction
Load data from all available record sets into DataFrames for analysis. Use the record set and field `@id`s gathered in the overview.

In [ ]:
# Extract data for each record set
dataframes = {}

for record_set_id in record_sets:
    records = list(dataset.records(record_set=record_set_id))
    if records:  # Only add DataFrame if records exist
        dataframes[record_set_id] = pd.DataFrame(records)
        print(f"\nLoaded DataFrame for record set @id: {record_set_id}")
        print("Columns:", dataframes[record_set_id].columns.tolist())
        print(dataframes[record_set_id].head())
    else:
        print(f"No records found for record set @id: {record_set_id}")

## 4. Exploratory Data Analysis (EDA)
Apply common data processing steps (filtering, normalization, grouping) to a record set and field.

We'll use a numeric field from the first loaded record set for demonstration. Make sure to reference by `@id`.

In [ ]:
# Select a record set and numeric field by their @id
if dataframes:
    # Pick first available record set and first numeric field
    record_set_id = list(dataframes.keys())[0]
    df = dataframes[record_set_id]

    # Identify numeric columns (float/int)
    numeric_fields = [col for col in df.columns if pd.api.types.is_numeric_dtype(df[col])]
    if numeric_fields:
        numeric_field_id = numeric_fields[0]    # Reference by field name
        print(f"Using numeric field @id: {numeric_field_id}")

        # Filtering example
        threshold = df[numeric_field_id].mean() if not pd.isnull(df[numeric_field_id].mean()) else 0
        filtered_df = df[df[numeric_field_id] > threshold]
        print(f"Filtered records with {numeric_field_id} > {threshold}:")
        print(filtered_df.head())

        # Normalization
        filtered_df[f"{numeric_field_id}_normalized"] = (
            filtered_df[numeric_field_id] - filtered_df[numeric_field_id].mean()
        ) / filtered_df[numeric_field_id].std()
        print(f"Normalized {numeric_field_id} for filtered records:")
        print(filtered_df[[numeric_field_id, f"{numeric_field_id}_normalized"]].head())

        # Grouping example: pick a categorical field
        categorical_fields = [col for col in df.columns if df[col].dtype == 'object' and col != numeric_field_id]
        if categorical_fields:
            group_field_id = categorical_fields[0]
            print(f"Grouping by field @id: {group_field_id}")
            grouped_df = filtered_df.groupby(group_field_id).mean(numeric_only=True)
            print(f"Grouped data by {group_field_id}:")
            print(grouped_df.head())
        else:
            print("No suitable categorical fields found for grouping.")
    else:
        print("No numeric fields available in record set.")
else:
    print("No dataframes loaded to perform EDA.")

## 5. Visualization
Visualize the distribution of a numeric field or the relationship between two fields in the dataset.
Reference fields by their `@id` (column names in the DataFrame).

In [ ]:
# Plotting the distribution of the numeric field
if dataframes and numeric_fields:
    plt.figure(figsize=(8, 4))
    df[numeric_field_id].dropna().hist(bins=20)
    plt.title(f"Distribution of numeric field: {numeric_field_id} (@id)")
    plt.xlabel(numeric_field_id)
    plt.ylabel("Frequency")
    plt.show()

    # Scatter plot (if there is another numeric field)
    if len(numeric_fields) > 1:
        plt.figure(figsize=(6, 4))
        plt.scatter(df[numeric_field_id], df[numeric_fields[1]])
        plt.title(f"Scatter Plot between {numeric_field_id} and {numeric_fields[1]} (@id)")
        plt.xlabel(numeric_field_id)
        plt.ylabel(numeric_fields[1])
        plt.show()
else:
    print("No numeric data available to plot.")

## 6. Conclusion
This notebook demonstrated how to:
- Load a Croissant dataset using `mlcroissant`
- Access and reference entities by `@id`
- Extract and review records from available record sets
- Apply common EDA techniques, such as filtering, normalization, and grouping
- Visualize numeric data distributions and relationships.

For deeper analysis, consult the Croissant schema documentation and explore individual field definitions by their `@id` for richer metadata.